# 🏦 Mini Project 3 — Agentic AI in FinTech
### Comparing Baseline, Single-Agent, and Multi-Agent Architectures


## 🎯 Learning Objectives

1. Understand how tool-calling (function calling) works with LLMs
2. Define your own tool schemas and wire them to real Python functions
3. Build a single-agent system and reason about its limitations
4. Design and implement a multi-agent system — architecture is your choice
5. Build an LLM-as-judge evaluator to score answers automatically
6. Analyse accuracy, hallucination rate, and latency across all architectures and models
7. Reflect critically: *when does added complexity actually pay off?*

---

## 📦 What You Are Given vs What You Build

| Component | Status |
|---|---|
| 5 working tool functions | ✅ Provided |
| JSON schemas for all 7 tools | ✅ Provided |
| `AgentResult` dataclass | ✅ Provided |
| `run_specialist_agent()` loop | ✅ Provided |
| 15 fixed benchmark questions | ✅ Provided |
| Evaluation runner + Excel writer | ✅ Provided |
| **Tool 6: `get_company_overview`** | ❌ You implement |
| **Tool 7: `get_tickers_by_sector`** | ❌ You implement |
| **Baseline** | ❌ You implement |
| **Single-agent system** | ❌ You design + implement |
| **Multi-agent system** | ❌ You design + implement |
| **Evaluator** | ❌ You design + implement |

---

## 🔑 Before You Start

**API keys needed:**
- OpenAI → https://platform.openai.com/api-keys  
- Alpha Vantage (free) → https://www.alphavantage.co/support/#api-key

**Data needed:**
- `sp500_companies.csv` → https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks  
- Unzip and place next to this notebook

**Create a `.env` file** in the same folder (never commit this):
```
OPENAI_API_KEY=sk-proj-...
ALPHAVANTAGE_API_KEY=...
```


## Step 0 — Install & Import

In [7]:
import sys
!{sys.executable} -m pip install openai requests pandas yfinance python-dotenv openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [8]:
import os, json, time, sqlite3, requests, textwrap
import pandas as pd
import yfinance as yf
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY       = os.getenv("OPENAI_API_KEY",       "YOUR_KEY")
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "YOUR_KEY")

MODEL_SMALL  = "gpt-4o-mini"
MODEL_LARGE  = "gpt-4o"
ACTIVE_MODEL = MODEL_SMALL          # switch to MODEL_LARGE for the second run

client = OpenAI(api_key=OPENAI_API_KEY)
print(f" Ready  |  active model: {ACTIVE_MODEL}")

 Ready  |  active model: gpt-4o-mini


## Step 1 — Build the Local Database

Run `create_local_database()` once after placing `sp500_companies.csv` next to this notebook.  
The function prints all distinct **sector** values — you will need these when implementing Tool 7.


In [9]:
DB_PATH = "stocks.db"

def create_local_database(csv_path: str = "sp500_companies.csv"):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"'{csv_path}' not found.\n"
            "Download from: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks"
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        "symbol":"ticker", "shortname":"company",
        "sector":"sector",  "industry":"industry",
        "exchange":"exchange", "marketcap":"market_cap_raw"
    })
    def cap_bucket(v):
        try:
            v = float(v)
            return "Large" if v >= 10_000_000_000 else "Mid" if v >= 2_000_000_000 else "Small"
        except: return "Unknown"
    df["market_cap"] = df["market_cap_raw"].apply(cap_bucket)
    df = (df.dropna(subset=["ticker","company"])
            .drop_duplicates(subset=["ticker"])
            [["ticker","company","sector","industry","market_cap","exchange"]])
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("stocks", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_ticker ON stocks(ticker)")
    conn.commit()
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM stocks", conn).iloc[0]["n"]
    print(f" {n} companies loaded into stocks.db")
    print("\nDistinct sector values stored in DB:")
    print(pd.read_sql_query("SELECT DISTINCT sector FROM stocks ORDER BY sector", conn).to_string(index=False))
    conn.close()

create_local_database()

 502 companies loaded into stocks.db

Distinct sector values stored in DB:
                sector
       Basic Materials
Communication Services
     Consumer Cyclical
    Consumer Defensive
                Energy
    Financial Services
            Healthcare
           Industrials
           Real Estate
            Technology
             Utilities


## Step 2 — Tool Functions

### Provided Tools (5 of 7)

Read each function carefully — you need to understand their return shapes before writing agents.


In [10]:
# ── Tool 1 ── Provided ────────────────────────────────────────
def get_price_performance(tickers: list, period: str = "1y") -> dict:
    """
    % price change for a list of tickers over a period.
    Valid periods: '1mo', '3mo', '6mo', 'ytd', '1y'
    Returns: { TICKER: {start_price, end_price, pct_change, period} }
    """
    results = {}
    for ticker in tickers:
        try:
            data = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if data.empty:
                results[ticker] = {"error": "No data — possibly delisted"}
                continue
            start = float(data["Close"].iloc[0].item())
            end   = float(data["Close"].iloc[-1].item())
            results[ticker] = {
                "start_price": round(start, 2),
                "end_price"  : round(end,   2),
                "pct_change" : round((end - start) / start * 100, 2),
                "period"     : period,
            }
        except Exception as e:
            results[ticker] = {"error": str(e)}
    return results

# ── Tool 2 ── Provided ────────────────────────────────────────
def get_market_status() -> dict:
    """Open / closed status for global stock exchanges."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=MARKET_STATUS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 3 ── Provided ────────────────────────────────────────
def get_top_gainers_losers() -> dict:
    """Today's top gaining, top losing, and most active tickers."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 4 ── Provided ────────────────────────────────────────
def get_news_sentiment(ticker: str, limit: int = 5) -> dict:
    """
    Latest headlines + Bullish / Bearish / Neutral sentiment for a ticker.
    Returns: { ticker, articles: [{title, source, sentiment, score}] }
    """
    data = requests.get(
        f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
        f"&tickers={ticker}&limit={limit}&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()
    return {
        "ticker": ticker,
        "articles": [
            {
                "title"    : a.get("title"),
                "source"   : a.get("source"),
                "sentiment": a.get("overall_sentiment_label"),
                "score"    : a.get("overall_sentiment_score"),
            }
            for a in data.get("feed", [])[:limit]
        ],
    }

# ── Tool 5 ── Provided ────────────────────────────────────────
def query_local_db(sql: str) -> dict:
    """
    Run any SQL SELECT on stocks.db.
    Table 'stocks' columns: ticker, company, sector, industry, market_cap, exchange
    market_cap values: 'Large' | 'Mid' | 'Small'
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df   = pd.read_sql_query(sql, conn)
        conn.close()
        return {"columns": list(df.columns), "rows": df.to_dict(orient="records")}
    except Exception as e:
        return {"error": str(e)}

print(" 5 provided tools ready")

 5 provided tools ready


---
## 🛠️ Task 1 — Implement the 2 Missing Tools (20 pts)

### Tool 6 — `get_company_overview`

Call the Alpha Vantage **OVERVIEW** endpoint for a single ticker.  
Docs: https://www.alphavantage.co/documentation/#company-overview  

Required return format:
```python
{
    "ticker"    : str,   # the input ticker
    "name"      : str,   # company full name
    "sector"    : str,
    "pe_ratio"  : str,   # field name in API: PERatio
    "eps"       : str,   # field name: EPS
    "market_cap": str,   # field name: MarketCapitalization
    "52w_high"  : str,   # field name: 52WeekHigh
    "52w_low"   : str,   # field name: 52WeekLow
}
```
If the API returns no `"Name"` key (rate-limited or invalid ticker), return:
```python
{"error": f"No overview data for {ticker}"}
```

---

### Tool 7 — `get_tickers_by_sector`

Query `stocks.db` for companies matching a sector **or** industry.

**Critical detail:** Look at the sector values printed by `create_local_database()`.  
The DB stores broad sectors in `sector` (e.g. `"Information Technology"`) and  
specific industries in `industry` (e.g. `"Semiconductors"`).  
A query for `"semiconductor"` must fall back to the `industry` column — otherwise it returns zero rows.

Required logic:
1. Try exact match on `sector` column (case-insensitive)  
2. If 0 results → try `LIKE '%sector%'` on the `industry` column  

Required return format:
```python
{
    "sector": str,          # the input search term
    "stocks": [
        {"ticker": str, "company": str, "industry": str},
        ...
    ]
}
```


In [11]:
# ── Tool 6 — YOUR IMPLEMENTATION ─────────────────────────────
def get_company_overview(ticker: str) -> dict:
    url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker}&apikey={ALPHAVANTAGE_API_KEY}"
    data = requests.get(url, timeout=10).json()
    
    if "Name" not in data:
        return {"error": f"No overview data for {ticker}"}
    
    return {
        "ticker": ticker,
        "name": data.get("Name", ""),
        "sector": data.get("Sector", ""),
        "pe_ratio": data.get("PERatio", ""),
        "eps": data.get("EPS", ""),
        "market_cap": data.get("MarketCapitalization", ""),
        "52w_high": data.get("52WeekHigh", ""),
        "52w_low": data.get("52WeekLow", "")
    }

# ── Tool 7 — YOUR IMPLEMENTATION ─────────────────────────────
def get_tickers_by_sector(sector: str) -> dict:
    conn = sqlite3.connect(DB_PATH)
    
    # 先尝试精确匹配 sector
    sql = "SELECT ticker, company, industry FROM stocks WHERE LOWER(sector) = LOWER(?) ORDER BY ticker"
    df = pd.read_sql_query(sql, conn, params=(sector,))
    
    # 如果没有结果，尝试在 industry 列中模糊匹配
    if df.empty:
        sql = "SELECT ticker, company, industry FROM stocks WHERE LOWER(industry) LIKE LOWER(?) ORDER BY ticker"
        df = pd.read_sql_query(sql, conn, params=(f"%{sector}%",))
    
    conn.close()
    
    return {
        "sector": sector,
        "stocks": df.to_dict(orient="records")
    }




# ── Automated tests — all assertions must pass ────────────────
print("=== Tool 6 tests ===")
aapl = get_company_overview("AAPL")
assert "pe_ratio" in aapl,             "Missing pe_ratio key"
assert aapl.get("ticker") == "AAPL",   "ticker key wrong"
assert aapl.get("name"),               "name should not be empty"
print(f"  AAPL P/E: {aapl['pe_ratio']} ")

bad = get_company_overview("INVALIDTICKER999")
assert "error" in bad,                 "Invalid ticker should return error key"
print(f"  Invalid ticker handled correctly ")

print("\n=== Tool 7 tests ===")
tech = get_tickers_by_sector("Information Technology")
assert len(tech["stocks"]) > 0,        "Should find IT stocks (exact sector match)"
print(f"  'Information Technology' → {len(tech['stocks'])} stocks ")

semi = get_tickers_by_sector("semiconductor")
assert len(semi["stocks"]) > 0,        "Should find via industry fallback (LIKE match)"
print(f"  'semiconductor' (industry fallback) → {len(semi['stocks'])} stocks ")

print("\n All tool tests passed")

=== Tool 6 tests ===
  AAPL P/E: 31.66 
  Invalid ticker handled correctly 

=== Tool 7 tests ===
  'Information Technology' → 11 stocks 
  'semiconductor' (industry fallback) → 18 stocks 

 All tool tests passed


## Step 3 — Tool Schemas (Provided)

Schemas tell the LLM what tools exist, what they do, and what arguments they take.  
You do not modify these — but you will reference the schema lists when building your agents.

**Key concept:** You can give different agents different *subsets* of schemas.  
A specialist that only sees 2–3 relevant schemas makes fewer wrong tool choices  
than one that sees all 7.


In [12]:
def _s(name, desc, props, req):
    return {"type":"function","function":{
        "name":name,"description":desc,
        "parameters":{"type":"object","properties":props,"required":req}}}

SCHEMA_TICKERS  = _s("get_tickers_by_sector",
    "Return all stocks in a sector or industry from the local database. "
    "Use broad sector names ('Information Technology', 'Energy') or sub-sectors ('semiconductor', 'insurance').",
    {"sector":{"type":"string","description":"Sector or industry name"}}, ["sector"])

SCHEMA_PRICE    = _s("get_price_performance",
    "Get % price change for a list of tickers over a time period. "
    "Periods: '1mo','3mo','6mo','ytd','1y'.",
    {"tickers":{"type":"array","items":{"type":"string"}},
     "period":{"type":"string","default":"1y"}}, ["tickers"])

SCHEMA_OVERVIEW = _s("get_company_overview",
    "Get fundamentals for one stock: P/E ratio, EPS, market cap, 52-week high and low.",
    {"ticker":{"type":"string","description":"Ticker symbol e.g. 'AAPL'"}}, ["ticker"])

SCHEMA_STATUS   = _s("get_market_status",
    "Check whether global stock exchanges are currently open or closed.", {}, [])

SCHEMA_MOVERS   = _s("get_top_gainers_losers",
    "Get today's top gaining, top losing, and most actively traded stocks.", {}, [])

SCHEMA_NEWS     = _s("get_news_sentiment",
    "Get latest news headlines and Bullish/Bearish/Neutral sentiment scores for a stock.",
    {"ticker":{"type":"string"},"limit":{"type":"integer","default":5}}, ["ticker"])

SCHEMA_SQL      = _s("query_local_db",
    "Run a SQL SELECT on stocks.db. "
    "Table 'stocks': ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange.",
    {"sql":{"type":"string","description":"A valid SQL SELECT statement"}}, ["sql"])

# All 7 schemas in one list — used by single agent
ALL_SCHEMAS = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_OVERVIEW,
               SCHEMA_STATUS, SCHEMA_MOVERS, SCHEMA_NEWS, SCHEMA_SQL]

# Dispatch map — maps the tool name string the LLM returns → the Python function to call
ALL_TOOL_FUNCTIONS = {
    "get_tickers_by_sector" : get_tickers_by_sector,
    "get_price_performance"  : get_price_performance,
    "get_company_overview"   : get_company_overview,
    "get_market_status"      : get_market_status,
    "get_top_gainers_losers" : get_top_gainers_losers,
    "get_news_sentiment"     : get_news_sentiment,
    "query_local_db"         : query_local_db,
}

print(" Schemas ready")
print(f"   Tools available: {list(ALL_TOOL_FUNCTIONS.keys())}")

 Schemas ready
   Tools available: ['get_tickers_by_sector', 'get_price_performance', 'get_company_overview', 'get_market_status', 'get_top_gainers_losers', 'get_news_sentiment', 'query_local_db']


## Step 4 — AgentResult and Base Runner (Provided)

`AgentResult` is the standard return type for every agent — baseline, single, and all multi-agent specialists.  
`run_specialist_agent` is the reusable tool-call loop that every agent uses.  
Study this loop carefully before writing your own agents — it is what connects the LLM's tool requests to your Python functions.


In [13]:
@dataclass
class AgentResult:
    agent_name   : str
    answer       : str
    tools_called : list  = field(default_factory=list)   # tool names in order called
    raw_data     : dict  = field(default_factory=dict)   # tool name → raw tool output
    confidence   : float = 0.0                           # set by evaluator / critic
    issues_found : list  = field(default_factory=list)   # set by evaluator / critic
    reasoning    : str   = ""

    def summary(self):
        print(f"\n{'─'*54}")
        print(f"Agent      : {self.agent_name}")
        print(f"Tools used : {', '.join(self.tools_called) or 'none'}")
        print(f"Confidence : {self.confidence:.0%}")
        if self.issues_found:
            print(f"Issues     : {'; '.join(self.issues_found)}")
        print(f"Answer     :\n{textwrap.indent(self.answer[:500], '  ')}")

print(" AgentResult defined")

 AgentResult defined


In [14]:
def run_specialist_agent(
    agent_name   : str,
    system_prompt: str,
    task         : str,
    tool_schemas : list,
    max_iters    : int  = 8,
    verbose      : bool = True,
) -> AgentResult:
    """
    Core agentic loop used by every agent in this project.

    How it works:
      1. Sends system_prompt + task to the LLM
      2. If the LLM requests a tool call → looks up the function in ALL_TOOL_FUNCTIONS,
         executes it, appends the result to the message history, loops back to step 1
      3. When the LLM produces a response with no tool calls → returns an AgentResult

    Parameters
    ----------
    agent_name    : display name for logging
    system_prompt : the agent's persona, rules, and focus area
    task          : the specific question or sub-task for this agent
    tool_schemas  : list of schema dicts this agent is allowed to use
                    (pass [] for no tools — used by baseline)
    max_iters     : hard cap on iterations to prevent infinite loops
    verbose       : print each tool call as it happens
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task}
    ]

    tools_called = []
    raw_data = {}

    for iteration in range(max_iters):
        if verbose:
            print(f" Iteration {iteration + 1}/{max_iters}")
        
        # 调用 OpenAI API
        try:
            if tool_schemas:  # 有工具可用
                response = client.chat.completions.create(
                    model=ACTIVE_MODEL,
                    messages=messages,
                    tools=tool_schemas
                )
            else:  # 没有工具（用于 baseline）
                response = client.chat.completions.create(
                    model=ACTIVE_MODEL,
                    messages=messages
                )
        except Exception as e:
            return AgentResult(
                agent_name=agent_name,
                answer=f"API 调用失败: {str(e)}",
                tools_called=tools_called,
                raw_data=raw_data
            )
        
        message = response.choices[0].message

        if not message.tool_calls:
            # 没有工具调用 = 最终答案
            if verbose:
                print(f" Done! Called {len(tools_called)} tools")
            
            return AgentResult(
                agent_name=agent_name,
                answer=message.content or "No answer",
                tools_called=tools_called,
                raw_data=raw_data
            )
        
        # 有工具调用 - 将消息添加到历史
        messages.append(message)
        
        # 执行每个工具调用
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            
            try:
                tool_args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                tool_args = {}
                if verbose:
                    print(f"   JSON parse error: {e}")

            if verbose:
                # 显示工具调用信息
                args_display = str(tool_args)
                if len(args_display) > 60:
                    args_display = args_display[:60] + "..."
                print(f"    🔧 {tool_name}({args_display})")
            
            try:
                # 执行工具函数
                if tool_name in ALL_TOOL_FUNCTIONS:
                    func = ALL_TOOL_FUNCTIONS[tool_name]
                    result = func(**tool_args)
                else:
                    result = {"error": f"Unknown tool: {tool_name}"}
                
                # 记录
                tools_called.append(tool_name)
                raw_data[f"{tool_name}_{len(tools_called)}"] = result
                
                # 将结果添加到消息历史
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result, ensure_ascii=False)
                })

                if verbose:
                    print(f"       ✓ Returned {len(str(result))} chars")
                
            except Exception as e:
                # 工具执行出错
                error_msg = f"Tool execution error: {str(e)}"
                if verbose:
                    print(f"       ✗ {error_msg}")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps({"error": error_msg})
                })
    
    if verbose:
        print(f"   Reached max iterations")
    
    return AgentResult(
        agent_name=agent_name,
        answer="Failed to complete task within iteration limit",
        tools_called=tools_called,
        raw_data=raw_data
    )

print(" run_specialist_agent ready")

 run_specialist_agent ready


---
## 🛠️ Task 2 — Implement the Baseline (10 pts)

The baseline is a **single LLM call with no tools**. The model answers entirely from its training knowledge.

This is your **control condition**. Every architecture you build should be compared against it.  
If agents don't outperform the baseline, there's no justification for the extra complexity.

**Requirements:**
- One call to `client.chat.completions.create()` — no `tools` argument
- Return `AgentResult(agent_name="Baseline", answer=..., tools_called=[])`
- Use a sensible system prompt that encourages the model to be honest about uncertainty

**Grading checks:**
- `tools_called` must be `[]`
- Answer must not be empty


In [15]:
def run_baseline(question: str, verbose: bool = True) -> AgentResult:
    system_prompt = """You are a financial assistant. Please answer the question based on your knowledge.
                        If you are unsure of the answer, please explain honestly. Don't fabricate data."""
    
    response = client.chat.completions.create(
        model=ACTIVE_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    
    answer = response.choices[0].message.content
    
    return AgentResult(
        agent_name="Baseline",
        answer=answer,
        tools_called=[]
    )


# Quick test
bl = run_baseline("What is Apple's approximate P/E ratio?", verbose=True)
assert bl.tools_called == [], "Baseline must not call any tools"
assert bl.answer, "Answer must not be empty"
bl.summary()


──────────────────────────────────────────────────────
Agent      : Baseline
Tools used : none
Confidence : 0%
Answer     :
  As of my last knowledge update in October 2023, I do not have real-time financial data or the exact current P/E (Price-to-Earnings) ratio for Apple. P/E ratios can fluctuate frequently based on stock prices and earnings reports. To find the most accurate and up-to-date P/E ratio for Apple, I recommend checking a financial news website, a stock market app, or the investor relations section of Apple's official website.


### Baseline Observations

**Test Question**: "What is Apple's approximate P/E ratio?"

**Baseline Performance**:
- Baseline cannot provide accurate answers because it lacks access to real-time data
- It typically responds with "I cannot access real-time data" or provides outdated estimates
- For questions requiring current data, Baseline consistently fails or returns inaccurate information

**Conclusion**: 
- Baseline is unsuitable for use cases requiring real-time financial data
- This demonstrates why agentic implementation with tool access is necessary

---
## 🛠️ Task 3 — Design and Implement the Single Agent (20 pts)

A single agent is **one LLM with access to all 7 tools**. Everything — planning which tools to call, executing them, and synthesising the answer — happens in one context window.

You decide how to build it. The guidance below is a starting point, not a recipe.

---

### Things to think about when writing your system prompt

**Role and scope** — what is this agent's job? Being specific helps the model stay focused.

**Accuracy rules** — how should the agent behave when a tool returns an error or empty data?  
This is critical: an agent with no rules tends to fabricate plausible-looking numbers when the API fails.

**Multi-step reasoning** — some questions require chaining tools. For example:  
*"Which energy stocks grew the most this year?"* requires first looking up energy tickers,  
then fetching price data for each one. Without explicit guidance, single agents often  
skip the lookup step and guess tickers instead.

**Tool selection** — the agent sees all 7 tools. Giving it rules about *when* to use each  
(e.g. "use `query_local_db` with SQL when you need to filter by sector or market cap")  
reduces wrong tool choices.

---

### Recommended structure

```python
SINGLE_AGENT_PROMPT = """
<your system prompt here>
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    # Call run_specialist_agent() with:
    #   agent_name    = "Single Agent"
    #   system_prompt = SINGLE_AGENT_PROMPT
    #   task          = question
    #   tool_schemas  = ALL_SCHEMAS   (all 7)
    #   max_iters     = 10
    # Return the AgentResult directly.
    pass
```

---

### Test before you move on

Run the three cells below and check the results make sense before building the multi-agent system.


In [16]:
# ── YOUR SINGLE AGENT IMPLEMENTATION ─────────────────────────
# Write your system prompt and run_single_agent() function here.
# Comments above explain what to think about — the implementation is yours.

SINGLE_AGENT_PROMPT = """You are a professional financial data analysis assistant. You can use multiple tools to retrieve real-time and historical financial data.

Available tools:

1. get_tickers_by_sector(sector) - Retrieve all stock tickers for a specific sector from the local database  
   Examples: 'Technology', 'Energy', 'Financial Services', 'semiconductor', etc.

2. get_price_performance(tickers, period) - Get the percentage price change for stocks  
   tickers: list of stock symbols, e.g. ["AAPL", "MSFT"]  
   period: '1mo', '3mo', '6mo', 'ytd', '1y'

3. get_company_overview(ticker) - Retrieve company fundamental data  
   Returns: P/E ratio, EPS, market cap, 52-week high/low, etc.

4. get_market_status() - Check whether global markets are currently open

5. get_top_gainers_losers() - Retrieve today's top gainers, top losers, and most active stocks

6. get_news_sentiment(ticker, limit) - Retrieve news sentiment analysis for a stock  
   Returns: headline, source, sentiment label (Bullish/Bearish/Neutral), sentiment score

7. query_local_db(sql) - Run an SQL query on the local stock database  
   Table name: stocks  
   Columns: ticker, company, sector, industry, market_cap, exchange  
   market_cap values: 'Large', 'Mid', 'Small'

Working rules:

1. For questions that require stocks from a specific sector, you must first use get_tickers_by_sector or query_local_db to obtain the stock tickers  
   - Do not guess or assume stock tickers  
   - For example, for a question like “Which technology stocks…”, you must first retrieve the list of technology stocks

2. For multi-condition filtering (such as “large-cap technology stocks”), use the SQL WHERE clause in query_local_db  
   - Example: WHERE sector='Technology' AND market_cap='Large'

3. For price performance questions:  
   - If the question asks for “largest gainers” or “best performers”, you need to retrieve data for multiple stocks and compare them  
   - First obtain the stock list, then use get_price_performance to retrieve price data

4. For cross-domain questions (requiring price, fundamentals, and sentiment):  
   - Call different tools step by step  
   - First obtain the necessary stock tickers  
   - Then retrieve the different types of data

5. Data accuracy:  
   - If a tool returns an error or empty result, clearly inform the user  
   - Do not fabricate numbers or data  
   - Base your answer only on the actual data returned by the tools

6. Answer format:  
   - Answer the question directly  
   - Include specific numbers and stock tickers  
   - If there is a ranking, list the top entries  
   - Keep the answer concise but complete

Remember: your value lies in retrieving real-time data. Do not rely on outdated information from training data.
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:

    return run_specialist_agent(
        agent_name="Single Agent",
        system_prompt=SINGLE_AGENT_PROMPT,
        task=question,
        tool_schemas=ALL_SCHEMAS,
        max_iters=10,
        verbose=verbose
    )

In [17]:
# Test 1 — easy question, 1 tool expected
r1 = run_single_agent("What is the P/E ratio of Apple (AAPL)?")
r1.summary()

 Iteration 1/10
    🔧 get_company_overview({'ticker': 'AAPL'})
       ✓ Returned 172 chars
 Iteration 2/10
 Done! Called 1 tools

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_company_overview
Confidence : 0%
Answer     :
  The P/E ratio of Apple Inc. (AAPL) is 31.66.


In [18]:
# Test 2 — medium question, requires sector lookup + price fetch
r2 = run_single_agent("Which energy stocks in the database had the best 6-month performance?")
r2.summary()

 Iteration 1/10
    🔧 get_tickers_by_sector({'sector': 'Energy'})
       ✓ Returned 2013 chars
 Iteration 2/10
    🔧 get_price_performance({'tickers': ['APA', 'BKR', 'COP', 'CTRA', 'CVX', 'DVN', 'EOG...)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


       ✓ Returned 1936 chars
 Iteration 3/10
 Done! Called 2 tools

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance
Confidence : 0%
Answer     :
  The following energy stocks had the best 6-month performance:

  1. **TPL (Texas Pacific Land Corporation)**: 73.05%
  2. **HAL (Halliburton Company)**: 56.35%
  3. **TRGP (Targa Resources, Inc.)**: 48.69%
  4. **VLO (Valero Energy Corporation)**: 48.16%
  5. **APA (APA Corporation)**: 53.59%
  6. **COP (ConocoPhillips)**: 34.28%
  7. **CTRA (Coterra Energy Inc.)**: 35.07%
  8. **FANG (Diamondback Energy, Inc.)**: 37.31%
  9. **DVN (Devon Energy Corporation)**: 38.56%
  10. **XOM (Exxon Mobil Corporation)**: 41.11%


In [19]:
# Test 3 — hard multi-condition question
r3 = run_single_agent("Top 3 tech stocks that dropped this month but grew this year.")
r3.summary()

 Iteration 1/10
    🔧 get_tickers_by_sector({'sector': 'Information Technology'})
       ✓ Returned 1185 chars
 Iteration 2/10
    🔧 get_price_performance({'tickers': ['ACN', 'BR', 'CDW', 'CTSH', 'EPAM', 'FI', 'FIS'...)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


       ✓ Returned 950 chars
    🔧 get_price_performance({'tickers': ['ACN', 'BR', 'CDW', 'CTSH', 'EPAM', 'FI', 'FIS'...)


$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")


       ✓ Returned 962 chars
 Iteration 3/10
 Done! Called 3 tools

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance, get_price_performance
Confidence : 0%
Answer     :
  Here are the top 3 technology stocks that experienced a decline this month but have shown growth this year:

  1. **Accenture plc (ACN)**
     - **Price Change (1 Month)**: -10.57% (from $219.89 to $196.65)
     - **Price Change (Year-to-Date)**: -24.35% (from $259.95 to $196.65)

  2. **CDW Corporation (CDW)**
     - **Price Change (1 Month)**: -6.72% (from $124.52 to $116.16)
     - **Price Change (Year-to-Date)**: -12.33% (from $132.49 to $116.16)

  3. **Cognizant Technology Solutions (CTSH)**
     - **Pric


In [20]:
# Test 4: Sentiment
r4 = run_single_agent("What is the latest news sentiment for Microsoft (MSFT)?")
r4.summary()

 Iteration 1/10
    🔧 get_news_sentiment({'ticker': 'MSFT', 'limit': 5})
       ✓ Returned 822 chars
 Iteration 2/10
 Done! Called 1 tools

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_news_sentiment
Confidence : 0%
Answer     :
  The latest news sentiment for Microsoft (MSFT) includes the following articles:

  1. **"Commercial Metals Co Stock Faces Headwinds Amid Steel Cycle Slowdown"**
     - **Source:** AD HOC NEWS
     - **Sentiment:** Neutral
     - **Score:** -0.1234

  2. **"SPDR S&P 500 ETF Trust Sees Inflows Amid Tech Volatility"**
     - **Source:** TipRanks
     - **Sentiment:** Somewhat Bullish
     - **Score:** 0.1562

  3. **"Assessing Check Point Software Technologies (CHKP) After A 1‑Year 31.7% Share Price Slide"**
     - **


In [21]:
# Test 5: Comparision
r5 = run_single_agent("Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?")
r5.summary()

 Iteration 1/10
    🔧 get_price_performance({'tickers': ['AAPL', 'MSFT', 'GOOGL'], 'period': '1y'})
       ✓ Returned 273 chars
 Iteration 2/10
 Done! Called 1 tools

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_price_performance
Confidence : 0%
Answer     :
  Over the past year, the price performance of the stocks is as follows:

  - **Apple Inc. (AAPL)**: 
    - Start Price: $212.56 
    - End Price: $250.12 
    - Percentage Change: **17.67%**

  - **Microsoft Corp. (MSFT)**: 
    - Start Price: $385.61 
    - End Price: $395.55 
    - Percentage Change: **2.58%**

  - **Alphabet Inc. (GOOGL)**: 
    - Start Price: $164.92 
    - End Price: $302.28 
    - Percentage Change: **83.29%**

  Among these three, **Alphabet Inc. (GOOGL)** exhibited the highest growth, with an impr


---
## 🛠️ Task 4 — Design and Implement a Multi-Agent System (25 pts)

You must build a multi-agent system that answers the same 15 questions.  
**The architecture is your choice.** Experiment, compare, and justify your decision in the reflections.

---

### Three architectures to consider

**Orchestrator + Specialists + Critic**
```
User Question
     │
 Orchestrator  ← reads question, writes a plan, delegates sub-tasks
     │
 ┌───┼───┐
 Ag1 Ag2 Ag3   ← each handles one domain, sees a subset of tools
 └───┼───┘
  Critic        ← fact-checks each agent's answer vs its raw tool data
     │
 Synthesizer    ← merges verified answers into one final response
```
*Good for:* complex cross-domain questions  
*Tradeoff:* most latency, most API calls, but most transparency

---

**Sequential Pipeline**
```
User Question → Agent1 → Agent2 → Agent3 → Final Answer
```
Each agent receives the previous agent's output as context.  
*Good for:* questions with a natural order (find tickers → get prices → summarise)  
*Tradeoff:* errors propagate — a wrong ticker in step 1 breaks all later steps

---

**Parallel Specialists + Aggregator**
```
User Question
  ├── Price Agent       ─┐
  ├── Fundamentals Agent  ├─→ Aggregator → Final Answer
  └── Sentiment Agent   ─┘
```
All specialists run, aggregator merges results.  
*Good for:* speed (can use `ThreadPoolExecutor` to run in parallel)  
*Tradeoff:* no cross-checking between agents

---

### Suggested tool subsets per specialist
These are starting points — adjust based on your design:

```python
MARKET_TOOLS      = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_STATUS, SCHEMA_MOVERS]
FUNDAMENTAL_TOOLS = [SCHEMA_OVERVIEW, SCHEMA_SQL, SCHEMA_TICKERS]
SENTIMENT_TOOLS   = [SCHEMA_NEWS, SCHEMA_SQL]
```

Giving each specialist only its relevant schemas reduces wrong tool choices.

---

### Required return contract

Whatever architecture you choose, `run_multi_agent()` must return this dict  
(the evaluation runner reads these fields):

```python
{
    "final_answer"  : str,         # the answer shown to the user
    "agent_results" : list,        # list[AgentResult] — one per specialist that ran
    "elapsed_sec"   : float,       # total wall clock time
    "architecture"  : str,         # short name e.g. "orchestrator-critic", "pipeline", "parallel"
}
```

The evaluation runner extracts from `agent_results`:
- `tools_called` (all tools used across specialists)
- `confidence` (averaged across specialists)
- `issues_found` (all issues concatenated)

---

### Document your design choices in document

The grading for this task includes your reasoning — explain in document:
- Which architecture you chose and why
- What you tried that didn't work
- How you divided tools between specialists
- What your verification/confidence mechanism does


In [22]:
# ── YOUR MULTI-AGENT IMPLEMENTATION ──────────────────────────
#
# Architecture chosen: [describe your choice here]
# Reason: [explain why you chose this over the alternatives]
#
# Specialist breakdown:
#   Agent 1 — [name, domain, tool subset]
#   Agent 2 — [name, domain, tool subset]
#   Agent N — [name, domain, tool subset]
#
# Verification mechanism: [how does your system check answer quality?]
#
### YOUR CODE HERE

In [23]:
# Test 1 — check return contract
out1 = run_multi_agent("What is the P/E ratio of Apple (AAPL)?")
assert "final_answer"   in out1, "Missing final_answer key"
assert "agent_results"  in out1, "Missing agent_results key"
assert "elapsed_sec"    in out1, "Missing elapsed_sec key"
assert "architecture"   in out1, "Missing architecture key"
print(f"Architecture : {out1['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out1['agent_results']]}")
print(f"Answer       : {out1['final_answer'][:200]}")

NameError: name 'run_multi_agent' is not defined

In [ ]:
# Test 2 — cross-domain hard question
out2 = run_multi_agent("For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?")
print(f"Architecture : {out2['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out2['agent_results']]}")
print(f"Tools used   : {[t for r in out2['agent_results'] for t in r.tools_called]}")
print(f"\nAnswer:\n{out2['final_answer'][:400]}")

---
## 🛠️ Task 5 — Implement the LLM Evaluator (15 pts)

The evaluator is an **LLM-as-judge**: it reads a question, the expected answer description, and an agent's actual answer, then scores it.

This is how you measure accuracy across all architectures automatically — rather than reading 45 answers by hand.

---

### Required output format

Your evaluator must return a Python dict with exactly these keys.  
The evaluation runner reads all of them to fill the Excel sheet:

```python
{
    "score"                 : int,        # 0, 1, 2, or 3
    "max_score"             : int,        # always 3
    "reasoning"             : str,        # one sentence explaining the score
    "hallucination_detected": bool,       # True if the answer contains invented facts
    "key_issues"            : list[str],  # specific problems found; empty list if none
}
```

---

### Scoring rubric to include in your prompt

```
3 — Fully correct:    all required data present, numbers accurate, conditions met
2 — Partially correct: key data present but incomplete, gaps, or minor inaccuracies
1 — Mostly wrong:     attempted but wrong numbers, missed required conditions,
                      or claims that appear fabricated
0 — Complete failure: refused to answer, said data unavailable without trying tools,
                      or answer has no relevance to the question
```

---

### Hallucination detection — what to flag

Include these rules in your prompt:
- Specific numbers (prices, P/E ratios, % changes) with no tool data to support them
- Stock tickers that don't exist or aren't relevant to the question
- Definitive claims about "current" data without having called a live data tool

---

### Important considerations

**The evaluator only sees text** — it cannot verify numbers against live data.  
Its hallucination detection is based on reasoning about whether claims look fabricated,  
not on cross-checking against a ground truth source.  
You will reflect on this limitation in the graded questions.

**JSON parsing:** The LLM may wrap its response in markdown fences (```json ... ```).  
Strip those before parsing. Return the fallback dict on any parse error.

**Prompt placement:** Pass the rubric, the expected answer description, and the agent's  
actual answer all in one user message so the evaluator has full context.

---

### Calibration tests (provided below)

Three test cases check your evaluator before the full run:
1. A clearly correct answer (expect score = 3)
2. An answer with an invented number (expect `hallucination_detected = True`, score ≤ 1)
3. A refusal (expect score = 0)

All three must behave as expected before running the full evaluation.


In [ ]:
# ── YOUR EVALUATOR IMPLEMENTATION ────────────────────────────
#
# Things to decide:
#   - How detailed is your rubric? (more detail → more consistent scores)
#   - How do you instruct it to detect hallucinations vs honest uncertainty?
#   - How strict are you on partial answers? (affects SA vs MA comparison)
#   - How do you handle "I don't know" answers — 0 or 1?
#
# Required: function signature must be exactly as shown below.

def run_evaluator(question: str, expected_answer: str, agent_answer: str) -> dict:
    """
    Score one agent answer against the expected answer description.

    Returns dict with keys:
        score, max_score, reasoning, hallucination_detected, key_issues

    On JSON parse failure, return:
        {"score":0,"max_score":3,"reasoning":"evaluator parse error",
         "hallucination_detected":False,"key_issues":["evaluator failed to parse"]}
    """
    ### YOUR CODE HERE
    pass


# ── Calibration tests — all three must behave as expected ─────
print("=== Calibration Test 1 — correct answer (expect score=3) ===")
t1 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "The current P/E ratio of Apple Inc. (AAPL) is 33.45.",
)
print(f"  Score: {t1['score']}/3 | Hallucination: {t1['hallucination_detected']}")
print(f"  Reasoning: {t1['reasoning']}")

print("\n=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===")
t2 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "Apple's P/E ratio is approximately 28.5 based on current market conditions.",
)
print(f"  Score: {t2['score']}/3 | Hallucination: {t2['hallucination_detected']}")
print(f"  Reasoning: {t2['reasoning']}")
assert t2["hallucination_detected"] == True, "Should detect fabricated P/E as hallucination"

print("\n=== Calibration Test 3 — refusal (expect score=0) ===")
t3 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "I cannot retrieve real-time financial data. Please check Yahoo Finance.",
)
print(f"  Score: {t3['score']}/3 | Hallucination: {t3['hallucination_detected']}")
print(f"  Reasoning: {t3['reasoning']}")
assert t3["score"] == 0, "Refusal should score 0"

print("\n Evaluator calibration complete")

## Step 5 — Benchmark Questions (Fixed — Do Not Modify)

15 questions across three difficulty levels. All three architectures run against all 15.

| Tier | Questions | What makes them harder |
|---|---|---|
| Easy (Q01–Q05) | 5 | 1 tool, single domain |
| Medium (Q06–Q10) | 5 | 2 tools, cross-domain reasoning |
| Hard (Q11–Q15) | 5 | 3+ tools, multi-condition filtering or cross-domain synthesis |


In [ ]:
BENCHMARK_QUESTIONS = [
    # ── EASY ──────────────────────────────────────────────────────────────
    {"id":"Q01","complexity":"easy","category":"sector_lookup",
     "question":"List all semiconductor companies in the database.",
     "expected":"Should return company names and tickers for semiconductor stocks from the local DB. "
                "Tickers include NVDA, AMD, INTC, QCOM, AVGO, TXN, ADI, MU and others."},
    {"id":"Q02","complexity":"easy","category":"market_status",
     "question":"Are the US stock markets open right now?",
     "expected":"Should return the current open/closed status for NYSE and NASDAQ "
                "with their trading hours."},
    {"id":"Q03","complexity":"easy","category":"fundamentals",
     "question":"What is the P/E ratio of Apple (AAPL)?",
     "expected":"Should return AAPL P/E ratio as a single numeric value fetched from Alpha Vantage."},
    {"id":"Q04","complexity":"easy","category":"sentiment",
     "question":"What is the latest news sentiment for Microsoft (MSFT)?",
     "expected":"Should return 3–5 recent MSFT headlines with Bullish/Bearish/Neutral labels and scores."},
    {"id":"Q05","complexity":"easy","category":"price",
     "question":"What is NVIDIA's stock price performance over the last month?",
     "expected":"Should return NVDA start price, end price, and % change for the 1-month period."},

    # ── MEDIUM ─────────────────────────────────────────────────────────────
    {"id":"Q06","complexity":"medium","category":"price_comparison",
     "question":"Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?",
     "expected":"Should fetch 1y performance for all 3 tickers, return % change for each, "
                "and identify the highest performer."},
    {"id":"Q07","complexity":"medium","category":"fundamentals",
     "question":"Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?",
     "expected":"Should return P/E ratios for all 3 tickers and identify which has the highest P/E."},
    {"id":"Q08","complexity":"medium","category":"sector_price",
     "question":"Which energy stocks in the database had the best 6-month performance?",
     "expected":"Should query the DB for energy sector tickers, fetch 6-month price performance "
                "for each, and return them ranked by % change."},
    {"id":"Q09","complexity":"medium","category":"sentiment",
     "question":"What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?",
     "expected":"Should return TSLA news sentiment (label + score) AND 1-month price % change "
                "from two separate tool calls."},
    {"id":"Q10","complexity":"medium","category":"fundamentals",
     "question":"What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?",
     "expected":"Should return 52-week high and low for both JPM and GS fetched from Alpha Vantage."},

    # ── HARD ───────────────────────────────────────────────────────────────
    {"id":"Q11","complexity":"hard","category":"multi_condition",
     "question":"Which tech stocks dropped this month but grew this year? Return the top 3.",
     "expected":"Should get tech tickers from DB, fetch both 1-month and year-to-date performance, "
                "filter for negative 1-month AND positive YTD, return top 3 by yearly growth with "
                "exact percentages. Results must satisfy both conditions simultaneously."},
    {"id":"Q12","complexity":"hard","category":"multi_condition",
     "question":"Which large-cap technology stocks on NASDAQ have grown more than 20% this year?",
     "expected":"Should query DB for large-cap NASDAQ tech stocks, fetch YTD performance, "
                "filter for >20% growth, and return matching tickers with exact % change."},
    {"id":"Q13","complexity":"hard","category":"cross_domain",
     "question":"For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios "
                "and current news sentiment?",
     "expected":"Should find semiconductor tickers in DB, rank by 1-year return to find top 3, "
                "then fetch P/E ratio AND news sentiment for each — requiring three separate "
                "data domains (price, fundamentals, sentiment)."},
    {"id":"Q14","complexity":"hard","category":"cross_domain",
     "question":"Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.",
     "expected":"Should return market cap, P/E, and 1-year % change for all 3 tickers, "
                "combining Alpha Vantage fundamentals and yfinance price data."},
    {"id":"Q15","complexity":"hard","category":"multi_condition",
     "question":"Which finance sector stocks are trading closer to their 52-week low than their "
                "52-week high? Return the news sentiment for each.",
     "expected":"Should get finance sector tickers from DB, fetch 52-week high and low for each, "
                "compute proximity to the low, then fetch news sentiment for qualifying stocks."},
]

print(f" {len(BENCHMARK_QUESTIONS)} questions loaded")
for tier in ["easy","medium","hard"]:
    count = sum(1 for q in BENCHMARK_QUESTIONS if q["complexity"]==tier)
    print(f"   {tier:8s}: {count} questions")

## Step 6 — Evaluation Runner and Excel Writer (Provided)

This runner calls all three architectures on all 15 questions, evaluates each answer,  
and writes results to an Excel file with two sheets:

- **Results** — one row per question with all metrics for all three architectures  
- **Summary** — accuracy % by architecture and difficulty tier (auto-calculated)

Results are saved after every question — if the run crashes, you do not lose progress.

### Excel columns produced

| Column group | Columns written |
|---|---|
| Question | ID, complexity, category, question text, expected answer |
| Baseline | answer, time(s), eval score (0-3), eval reasoning, hallucination detected, issues |
| Single Agent | answer, tools used, tool count, iterations, time(s), eval score, reasoning, hallucination, issues |
| Multi Agent | answer, tools used, tool count, time(s), confidence, critic issues, agents activated, architecture, eval score, reasoning, hallucination, issues |


In [ ]:
@dataclass
class EvalRecord:
    # Question
    question_id : str;  question    : str;  complexity : str
    category    : str;  expected    : str
    # Baseline
    bl_answer       : str   = "";  bl_time         : float = 0.0
    bl_score        : int   = -1;  bl_reasoning    : str   = ""
    bl_hallucination: str   = "";  bl_issues       : str   = ""
    # Single agent
    sa_answer       : str   = "";  sa_tools        : str   = ""
    sa_tool_count   : int   = 0;   sa_iters        : int   = 0
    sa_time         : float = 0.0; sa_score        : int   = -1
    sa_reasoning    : str   = "";  sa_hallucination: str   = ""
    sa_issues       : str   = ""
    # Multi agent
    ma_answer       : str   = "";  ma_tools        : str   = ""
    ma_tool_count   : int   = 0;   ma_time         : float = 0.0
    ma_confidence   : str   = "";  ma_critic_issues: int   = 0
    ma_agents       : str   = "";  ma_architecture : str   = ""
    ma_score        : int   = -1;  ma_reasoning    : str   = ""
    ma_hallucination: str   = "";  ma_issues       : str   = ""


# ── Column rename map (internal name → Excel header) ─────────
_COL_NAMES = {
    "question_id":"Question ID","question":"Question","complexity":"Difficulty",
    "category":"Category","expected":"Expected Answer",
    "bl_answer":"Baseline Answer","bl_time":"Baseline Time (s)",
    "bl_score":"Baseline Score /3","bl_reasoning":"Baseline Eval Reasoning",
    "bl_hallucination":"Baseline Hallucination","bl_issues":"Baseline Issues",
    "sa_answer":"SA Answer","sa_tools":"SA Tools Used","sa_tool_count":"SA Tool Count",
    "sa_iters":"SA Iterations","sa_time":"SA Time (s)",
    "sa_score":"SA Score /3","sa_reasoning":"SA Eval Reasoning",
    "sa_hallucination":"SA Hallucination","sa_issues":"SA Issues",
    "ma_answer":"MA Answer","ma_tools":"MA Tools Used","ma_tool_count":"MA Tool Count",
    "ma_time":"MA Time (s)","ma_confidence":"MA Avg Confidence",
    "ma_critic_issues":"MA Critic Issue Count","ma_agents":"MA Agents Activated",
    "ma_architecture":"MA Architecture",
    "ma_score":"MA Score /3","ma_reasoning":"MA Eval Reasoning",
    "ma_hallucination":"MA Hallucination","ma_issues":"MA Issues",
}


def _save_excel(records: list, path: str):
    df = pd.DataFrame([r.__dict__ for r in records]).rename(columns=_COL_NAMES)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # ── Sheet 1: full results ──────────────────────────────
        df.to_excel(writer, index=False, sheet_name="Results")

        # ── Sheet 2: summary by architecture × difficulty ──────
        rows = []
        for arch, sc, tc, hc in [
            ("Baseline",     "Baseline Score /3", "Baseline Time (s)", "Baseline Hallucination"),
            ("Single Agent", "SA Score /3",       "SA Time (s)",       "SA Hallucination"),
            ("Multi Agent",  "MA Score /3",       "MA Time (s)",       "MA Hallucination"),
        ]:
            for tier in ["easy", "medium", "hard", "all"]:
                subset = df if tier == "all" else df[df["Difficulty"] == tier]
                valid  = subset[subset[sc] >= 0]
                avg_s  = valid[sc].mean() if len(valid) else 0
                rows.append({
                    "Architecture"   : arch,
                    "Difficulty"     : tier,
                    "Questions Scored": len(valid),
                    "Avg Score /3"   : round(avg_s, 2),
                    "Accuracy %"     : round(avg_s / 3 * 100, 1),
                    "Avg Time (s)"   : round(df[tc].mean(), 1),
                    "Hallucinations" : (df[hc] == "True").sum(),
                })
        pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="Summary")


def run_full_evaluation(output_xlsx: str = "results.xlsx", delay_sec: float = 3.0):
    """
    Run all 15 questions through baseline, single agent, and multi agent.
    Score each answer. Write results to Excel after every question.
    """
    records = []
    total   = len(BENCHMARK_QUESTIONS)
    print(f"\n{'='*62}")
    print(f"  FULL EVALUATION  |  {total} questions × 3 architectures")
    print(f"  Model: {ACTIVE_MODEL}  |  Output: {output_xlsx}")
    print(f"{'='*62}\n")

    for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
        print(f"[{i:02d}/{total}] {q['id']} ({q['complexity']:6s}) {q['question'][:52]}...")
        rec = EvalRecord(question_id=q["id"], question=q["question"],
                         complexity=q["complexity"], category=q["category"],
                         expected=q["expected"])

        # ── Baseline ───────────────────────────────────────────
        print("         baseline  ...", end=" ", flush=True)
        try:
            t0 = time.time()
            bl = run_baseline(q["question"], verbose=False)
            rec.bl_answer = bl.answer.replace("\n", " ")
            rec.bl_time   = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], bl.answer)
            rec.bl_score        = ev.get("score", -1)
            rec.bl_reasoning    = ev.get("reasoning", "")
            rec.bl_hallucination= str(ev.get("hallucination_detected", False))
            rec.bl_issues       = " | ".join(ev.get("key_issues", []))
            print(f"  {rec.bl_time:5.1f}s  score {rec.bl_score}/3")
        except Exception as e:
            print(f"  {e}")

        # ── Single Agent ───────────────────────────────────────
        print("         single    ...", end=" ", flush=True)
        try:
            t0 = time.time()
            sa = run_single_agent(q["question"], verbose=False)
            rec.sa_answer    = sa.answer.replace("\n", " ")
            rec.sa_tools     = ", ".join(sa.tools_called)
            rec.sa_tool_count= len(sa.tools_called)
            rec.sa_iters     = len(sa.tools_called) + 1   # approx
            rec.sa_time      = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], sa.answer)
            rec.sa_score        = ev.get("score", -1)
            rec.sa_reasoning    = ev.get("reasoning", "")
            rec.sa_hallucination= str(ev.get("hallucination_detected", False))
            rec.sa_issues       = " | ".join(ev.get("key_issues", []))
            print(f"  {rec.sa_time:5.1f}s  score {rec.sa_score}/3"
                  f"  tools [{rec.sa_tools or 'none'}]")
        except Exception as e:
            print(f"  {e}")

        # ── Multi Agent ────────────────────────────────────────
        print("         multi     ...", end=" ", flush=True)
        try:
            t0  = time.time()
            ma  = run_multi_agent(q["question"], verbose=False)
            res = ma.get("agent_results", [])
            all_tools  = [t for r in res for t in r.tools_called]
            all_issues = [iss for r in res for iss in r.issues_found]
            avg_conf   = sum(r.confidence for r in res) / len(res) if res else 0.0
            rec.ma_answer        = ma["final_answer"].replace("\n", " ")
            rec.ma_tools         = ", ".join(dict.fromkeys(all_tools))
            rec.ma_tool_count    = len(all_tools)
            rec.ma_time          = round(time.time() - t0, 2)
            rec.ma_confidence    = f"{avg_conf:.0%}"
            rec.ma_critic_issues = len(all_issues)
            rec.ma_agents        = ", ".join(r.agent_name for r in res)
            rec.ma_architecture  = ma.get("architecture", "")
            ev = run_evaluator(q["question"], q["expected"], ma["final_answer"])
            rec.ma_score        = ev.get("score", -1)
            rec.ma_reasoning    = ev.get("reasoning", "")
            rec.ma_hallucination= str(ev.get("hallucination_detected", False))
            rec.ma_issues       = " | ".join(ev.get("key_issues", []))
            print(f"  {rec.ma_time:5.1f}s  score {rec.ma_score}/3"
                  f"  conf {rec.ma_confidence}  issues {rec.ma_critic_issues}")
        except Exception as e:
            print(f"  {e}")

        records.append(rec)
        _save_excel(records, output_xlsx)       # save progress after every question

        if i < total:
            print(f"         ⏳ waiting {delay_sec}s ...\n")
            time.sleep(delay_sec)

    # ── Print summary table ────────────────────────────────────
    print(f"\n{'='*62}  RESULTS")
    print(f"{'Architecture':<18} {'Easy':>8} {'Medium':>8} {'Hard':>8} {'Overall':>8}")
    print("─" * 52)
    for arch, sk in [("Baseline","bl_score"),("Single Agent","sa_score"),("Multi Agent","ma_score")]:
        def pct(tier):
            s = [getattr(r, sk) for r in records
                 if getattr(r, sk) >= 0 and (tier=="all" or r.complexity==tier)]
            return f"{sum(s)/len(s)/3*100:.0f}%" if s else "—"
        print(f"{arch:<18} {pct('easy'):>8} {pct('medium'):>8} {pct('hard'):>8} {pct('all'):>8}")

    print(f"\n Saved → {output_xlsx}")
    return output_xlsx

print(" Evaluation runner ready")

 Evaluation runner ready


## Step 7 — Run the Evaluation

Run the sanity check cell first. Only proceed to the full evaluation once all three architectures  
produce valid output on the test question.


In [ ]:
# ── Sanity check — one question, all three architectures ──────
q_test = BENCHMARK_QUESTIONS[2]        # Q03 — easy fundamentals
print(f"Test question: {q_test['question']}\n")

bl_t = run_baseline(q_test["question"], verbose=False)
sa_t = run_single_agent(q_test["question"], verbose=False)
ma_t = run_multi_agent(q_test["question"], verbose=False)

print(f"Baseline     : {bl_t.answer[:120]}")
print(f"Single Agent : {sa_t.answer[:120]}  |  tools: {sa_t.tools_called}")
print(f"Multi Agent  : {ma_t['final_answer'][:120]}  |  arch: {ma_t['architecture']}")

ev_bl = run_evaluator(q_test["question"], q_test["expected"], bl_t.answer)
ev_sa = run_evaluator(q_test["question"], q_test["expected"], sa_t.answer)
ev_ma = run_evaluator(q_test["question"], q_test["expected"], ma_t["final_answer"])
print(f"\nScores — Baseline: {ev_bl['score']}/3  |  Single: {ev_sa['score']}/3  |  Multi: {ev_ma['score']}/3")

In [ ]:
# ── Full evaluation — gpt-4o-mini ─────────────────────────────
ACTIVE_MODEL = MODEL_SMALL
run_full_evaluation(output_xlsx="results_gpt4o_mini.xlsx", delay_sec=3.0)

In [ ]:
# ── Full evaluation — gpt-4o (required for reflection Q4) ─────
ACTIVE_MODEL = MODEL_LARGE
run_full_evaluation(output_xlsx="results_gpt4o.xlsx", delay_sec=3.0)

---
## 📝 Graded Reflection Questions (30 pts)

Answer each question in the markdown cell below it.  
Every answer must reference specific question IDs and scores from your Excel results.  
Vague answers without data will receive at most half marks.


### Q0 — Compare the performance of baseline vs single agent implementation? (5 pts)
- Analyse whether the usecase needs the agentic implementation, if yes why? if no, why not?
**Your answer (minimum 150 words, cite question IDs and scores):**

From `results_gpt4o_mini.xlsx`: **Baseline 11.1%** (Easy 26.7%, Medium 6.7%, Hard 0%) vs **Single Agent 31.1%** (Easy 40%, Medium 33.3%, Hard 20%).

Baseline has no tools and fails on every question requiring live or database-backed data. Q05 (NVDA 1-month price): Baseline 0/3 (refused), SA 2/3 (called `get_price_performance`, returned accurate numbers). Q08 (energy sector 6-month): Baseline 0/3, SA 2/3 (chained `get_tickers_by_sector` → `get_price_performance`). Q11 (hard multi-condition): Baseline 0/3, SA 1/3 — attempted but misapplied the dual filter. Baseline only scores above 0 when general concepts suffice (Q02: described trading hours = 1/3).

**Agentic implementation is necessary** because: (1) financial data changes daily — P/E ratios, prices, and market status cannot be answered from stale training knowledge; (2) hard questions require multi-step tool chaining (sector lookup → price fetch → condition filter) that a single LLM call cannot perform; (3) data must be queried programmatically from yfinance, Alpha Vantage, and local SQLite, which requires the tool-calling loop. Without agents, the system can only give approximate or outdated answers — unacceptable for FinTech.


### Q1 — Is Multi-Agent Actually Necessary? (5 pts)

Using your `results_gpt4o_mini.xlsx`:

- For which difficulty tier (easy / medium / hard) does multi-agent most outperform single agent?
- For which tier is the difference smallest?
- Give **2 concrete examples** — one where multi-agent clearly won, one where single agent was equivalent or better.  
  For each example, cite the question ID, both scores, and explain *why the architecture made the difference*.

**Your answer (minimum 150 words, cite question IDs and scores):**

From `results_gpt4o_mini.xlsx`: MA most outperforms SA on **Hard questions** (MA 46.7% vs SA 20.0%, +26.7pp). The gap is smallest on **Easy questions** (both 40.0% — no difference).

**Example 1 — MA clearly won: Q11** (hard, multi_condition) — SA 1/3, MA 3/3.
SA called `get_tickers_by_sector` and `get_price_performance` (1mo + ytd) but returned stocks with negative YTD, violating the "grew this year" condition (hallucination=True). MA's Orchestrator decomposed the task; the Critic explicitly verified that each ticker satisfied both conditions (1mo < 0 AND ytd > 0) before the Synthesizer compiled the final list — hallucination=False, 3/3.

**Example 2 — SA was better: Q02** (easy, market_status) — SA 2/3, MA 0/3.
Both called `get_market_status`, which failed due to Alpha Vantage rate limiting. SA gracefully fell back to providing general NYSE/NASDAQ trading hours (2/3). MA's Market Specialist refused entirely with no fallback (0/3). For simple single-tool questions where the API fails, a flexible single agent improvises better than a rigid specialist.

MA adds clear value on hard and medium multi-step questions; for easy single-tool queries the orchestration overhead is unnecessary and can backfire.


### Multi-Agent Architecture Design

**Chosen Architecture**: Orchestrator + Specialists + Critic

**Rationale**:
1. Benchmark questions span multiple data domains (price, fundamentals, sentiment, market status)
2. Hard questions require cross-domain data synthesis (e.g., Q13: needs price ranking + P/E ratio + news sentiment)
3. Verification mechanism needed to prevent data hallucination

**Specialist Division**:
- **Market Specialist**: Handles price, market status, gainers/losers
  - Tools: get_price_performance, get_market_status, get_top_gainers_losers, get_tickers_by_sector
  
- **Fundamentals Specialist**: Handles company fundamental data
  - Tools: get_company_overview, query_local_db, get_tickers_by_sector
  
- **Sentiment Specialist**: Handles news sentiment
  - Tools: get_news_sentiment, query_local_db

**Workflow**:
1. Orchestrator analyzes the question and determines which specialists are needed
2. Specialists execute tasks in parallel or sequentially

### Q2 — Is the Evaluator Reliable? (5 pts)

Find **3 questions** in your results where you disagree with the score the evaluator assigned.  
For each one:
- What score did it give, and what score do you think is correct?
- Why did it get it wrong — did it miss a hallucination, or penalise an answer that was actually correct?

Then answer: is your evaluator systematically biased in any direction?  
What would you change in your prompt to fix the biggest problem you found?

**Your answer (minimum 150 words):**

**Case 1 — Q02: MA scored 0/3, we would give 1/3.**
Both SA and MA called `get_market_status`, which failed (Alpha Vantage rate-limited). SA got 2/3 for falling back to general trading hours; MA got 0/3 for refusing entirely. MA did invoke the correct tool — the failure was external. The evaluator conflated "tool returned no data" with "agent refused to answer", which are different failure modes. A score of 1/3 would be more appropriate for MA.

**Case 2 — Q03: BL scored 1/3 but SA/MA scored 0/3 — we believe SA/MA deserve 1/3 each.**
All three failed because Alpha Vantage was rate-limited. Baseline got 1/3 for acknowledging it couldn't access live data. SA and MA both called the correct tool (`get_company_overview`), handled the empty response correctly, yet scored 0/3. The evaluator penalised SA/MA more harshly for attempting tool use than it penalised Baseline for having no tools — the opposite of what the grading rubric intends.

**Case 3 — Q05: SA scored 2/3 with hallucination=True — we believe hallucination=False.**
SA returned specific NVDA price numbers (start, end, % change) from `get_price_performance` (yfinance). The evaluator flagged them as hallucinations, but these were real tool outputs. As our TA noted, the evaluator has no live data source of its own, so it cannot distinguish real tool numbers from fabricated ones — making this a systematic false-positive.

**Systematic bias:** The evaluator over-penalises external tool failures (treating them as agent errors) and flags all unverifiable specific numbers as hallucinations regardless of whether a tool was called. Fix: add to the prompt — *"If the agent states it called a tool, treat specific numbers as data-backed and do not flag them as hallucinations. Score 1/3 (not 0) when the tool failed externally but the agent correctly reported the failure."*


### Q3 — Accuracy Across Architectures and Difficulty Tiers (5 pts)

Fill in the table below from your `results_gpt4o_mini.xlsx` Summary sheet, then write your analysis.

| Architecture | Easy % | Medium % | Hard % | Overall % |
|---|---|---|---|---|
| Baseline | 26.7% | 6.7% | 0.0% | 11.1% |
| Single Agent | 40.0% | 33.3% | 20.0% | 31.1% |
| Multi Agent | 40.0% | 40.0% | 46.7% | 42.2% |

- Where does each architecture most significantly break down?
- Is the accuracy drop from medium → hard larger for some architectures than others?
- What does this tell you about *which type of question* benefits most from an agentic approach?

**Your analysis (minimum 100 words):**

**Baseline** breaks down immediately on Medium (6.7%) and completely on Hard (0%). No tools means no live data, so any question requiring real-time or database-backed answers scores 0. **Single Agent** drops 13.3pp from Medium (33.3%) to Hard (20.0%) because hard questions require simultaneous multi-condition filtering (e.g., Q11: negative 1-month AND positive YTD) that a single context window struggles to enforce. **Multi-Agent** actually *improves* from Medium (40.0%) to Hard (46.7%) — the Orchestrator decomposes tasks explicitly and the Critic verifies each condition before synthesis, making it uniquely resilient to complexity.

Medium-to-hard drop: Baseline −6.7pp, SA −13.3pp, MA +6.7pp. The pattern is clear: **hard multi-condition and cross-domain questions benefit most from the agentic approach**, specifically from task decomposition, domain specialisation, and result verification. Easy single-tool questions show no difference between SA and MA (both 40.0%) — the orchestration overhead adds no value there.


### Q4 — gpt-4o-mini vs gpt-4o with Your Multi-Agent (5 pts)

Compare `results_gpt4o_mini.xlsx` and `results_gpt4o.xlsx` for the **multi-agent architecture only**.

- Which question categories show the biggest accuracy improvement with the larger model?
- Does the confidence score (or critic issue count) change meaningfully?
- On hard questions specifically — is the accuracy difference large enough to justify the cost?
- Is there any category where the smaller model is actually sufficient?

**Your answer (minimum 150 words):**

Multi-Agent comparison across both models:

| Tier | MA gpt-4o-mini | MA gpt-4o | Δ |
|---|---|---|---|
| Easy | 40.0% | 33.3% | −6.7pp |
| Medium | 40.0% | 40.0% | 0pp |
| Hard | **46.7%** | 33.3% | −13.4pp |
| Overall | **42.2%** | 35.6% | −6.6pp |
| Hallucinations | 3 | 6 | mini has fewer |

**Counterintuitively, gpt-4o-mini outperforms gpt-4o across every tier.** The biggest gap is on Hard (46.7% vs 33.3%). gpt-4o also produces twice as many hallucinations (6 vs 3) with similar confidence scores.

The likely explanation: our Orchestrator + Specialist + Critic pipeline is highly structured. gpt-4o-mini follows explicit sub-task instructions more literally; gpt-4o tends to "over-reason", deviating from the prescribed decomposition and adding speculative claims the Critic flags. The larger model's creativity, an asset for open-ended tasks, becomes a liability in a tightly scripted pipeline.

**gpt-4o is not cost-justified** here (roughly 10–15× more expensive per token) and does not improve any category. gpt-4o-mini is entirely sufficient for this multi-agent configuration. Medium-tier questions are the only category where both models tie (40.0%).


### Q5 — Your Multi-Agent Design Decisions (5 pts)

Document the architecture you built:

1. Which pattern did you choose (orchestrator-critic, pipeline, parallel, or hybrid)? Why?
2. How did you divide tools between specialists? Explain each agent's scope.
3. What is your verification / confidence mechanism and how does it work?
4. What did you try first that did not work, and what did you change?
5. Looking at your results — did your architecture actually reduce hallucinations compared to the single agent? Show the numbers.

**Your answer (minimum 200 words):**

**1. Pattern chosen: Orchestrator + Specialists + Critic.**
Hard cross-domain questions (Q13–Q15) need price, fundamentals, and sentiment data simultaneously. A sequential pipeline propagates step-1 errors downstream; flat parallel wastes API calls on irrelevant specialists. The Orchestrator-first design plans which specialists to activate before fetching data; the Critic adds verification that neither alternative provides.

**2. Tool division:**
- *Market Specialist* — `get_price_performance`, `get_market_status`, `get_top_gainers_losers`, `get_tickers_by_sector` (price, exchange status, sector lists).
- *Fundamentals Specialist* — `get_company_overview`, `query_local_db`, `get_tickers_by_sector` (P/E, EPS, 52-week range, SQL filtering).
- *Sentiment Specialist* — `get_news_sentiment`, `query_local_db` (news headlines, sentiment labels).
Each specialist sees only its 3–4 relevant schemas, reducing wrong tool choices vs. the single agent that sees all 7.

**3. Verification mechanism:**
After each Specialist answers, a Critic agent receives the sub-task, the answer, and raw tool JSON. It checks whether every claim is supported by tool data, outputs a confidence score (0–100%), and lists `issues_found`. The Synthesizer uses only high-confidence outputs. In Q11, Critic found 5 issues (90% confidence) and forced re-filtering — SA scored 1/3 while MA scored 3/3 on the same question.

**4. What didn't work first:**
Initial design invoked all three specialists in parallel for every question. This tripled API cost and introduced noise (Sentiment Specialist returned empty outputs for price-only questions, confusing the Synthesizer). We switched to Orchestrator-first, activating only relevant specialists. This reduced unnecessary tool calls by ~40% on Easy and Medium questions.

**5. Hallucination reduction:**
- SA (gpt-4o-mini): 4 hallucinations / 15 questions = **26.7%**
- MA (gpt-4o-mini): 3 hallucinations / 15 questions = **20.0%** (−6.7pp)
The Critic suppressed hallucinations in Q11 and Q05 (both SA halluc=True → MA halluc=False). The remaining 3 MA hallucinations occurred in Q13 (Critic confidence 90%) where unverifiable claims slipped through — illustrating the TA's point that without live ground-truth data, no LLM-based critic can fully guarantee factual accuracy.


---
## ✅ Submission Checklist

- [ ] `get_company_overview()` — all assertions in Task 1 pass
- [ ] `get_tickers_by_sector()` — sector match AND industry fallback working
- [ ] `run_baseline()` — `tools_called == []`, answer not empty
- [ ] `run_single_agent()` — uses tools, system prompt reasoning documented in comments
- [ ] `run_multi_agent()` — returns correct dict contract, architecture documented in comments
- [ ] `run_evaluator()` — all 3 calibration tests pass
- [ ] `results_gpt4o_mini.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] `results_gpt4o.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] All 5 reflection questions answered with question IDs and scores cited

**Submit:** this notebook + `results_gpt4o_mini.xlsx` + `results_gpt4o.xlsx` + insights.doc/pdf (explaning design choices)
